# 02 Preprocesado

Aqui dejo preparado el dataset que despues uso en los tres apartados.


## Objetivo

Quiero separar la limpieza comun de los datasets concretos de clasificacion, regresion y no supervisado.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocess_base import (
    cargar_data,
    limpieza_basica,
    permutar_filas,
    separa_tipos,
    tratar_ceros_notas_sin_evaluacion,
)
from src.preprocess_clasificacion import get_classification_dataset
from src.preprocess_regresion import get_regression_dataset
from src.preprocess_no_supervisado import get_unsupervised_dataset

pd.set_option("display.max_columns", 100)

In [2]:
df_raw = cargar_data()
print("Shape raw:", df_raw.shape)
df_raw.head()

Shape raw: (4424, 37)
Out[2]: 
  estado_civil                         modo_solicitud  orden_solicitud  \
0    soltero/a            2a_fase_contingente_general                5   
1    soltero/a  estudiante_internacional_licenciatura                1   
2    soltero/a            1a_fase_contingente_general                5   
3    soltero/a            2a_fase_contingente_general                2   
4     casado/a                       mayor_de_23_anos                1   

                           curso asistencia_diurna_vespertina  \
0  animacion_y_diseno_multimedia                       diurna   
1                        turismo                       diurna   
2         diseno_de_comunicacion                       diurna   
3      periodismo_y_comunicacion                       diurna   
4       servicio_social_nocturno                   vespertina   

   cualificacion_previa  nota_cualificacion_previa nacionalidad  \
0  educacion_secundaria                      122.0   portuguesa   

## Limpieza basica

La limpieza base es sencilla: cambia `en_blanco` a missing y permite decidir que hacer con `desconocido`.


In [3]:
df_base_unknown_cat = limpieza_basica(df_raw, desconocido_es_nan=False)
df_base_unknown_nan = limpieza_basica(df_raw, desconocido_es_nan=True)

comparacion_unknown = pd.DataFrame(
    {
        "missing_sin_tocar_desconocido": df_base_unknown_cat.isna().sum(),
        "missing_con_desconocido_nan": df_base_unknown_nan.isna().sum(),
    }
)
comparacion_unknown["incremento_missing"] = (
    comparacion_unknown["missing_con_desconocido_nan"]
    - comparacion_unknown["missing_sin_tocar_desconocido"]
)
comparacion_unknown[comparacion_unknown["incremento_missing"] > 0]

Out[3]: 
                     missing_sin_tocar_desconocido  \
cualificacion_madre                              0   
cualificacion_padre                              0   

                     missing_con_desconocido_nan  incremento_missing  
cualificacion_madre                          130                 130  
cualificacion_padre                          112                 112  


### Decision sobre `desconocido`

Mantengo `desconocido` como categoria. Puede ser ausencia de informacion, pero tambien puede tener informacion propia.


In [4]:
df_clean = limpieza_basica(df_raw, desconocido_es_nan=False)

resumen_especiales = pd.DataFrame(
    {
        "en_blanco_restante": (df_clean == "en_blanco").sum(),
        "desconocido_restante": (df_clean == "desconocido").sum(),
    }
)
resumen_especiales[(resumen_especiales["en_blanco_restante"] > 0) | (resumen_especiales["desconocido_restante"] > 0)]

Out[4]: 
                     en_blanco_restante  desconocido_restante
cualificacion_madre                   0                   130
cualificacion_padre                   0                   112


## Ceros en notas

Si una nota es 0 y no hay evaluaciones, la paso a missing. Si hay evaluaciones, mantengo el 0 como posible nota real.


In [5]:
antes = pd.DataFrame(
    {
        "zeros_1sem": [int((df_clean["nota_media_1sem"] == 0).sum())],
        "zeros_2sem": [int((df_clean["nota_media_2sem"] == 0).sum())],
        "na_1sem": [int(df_clean["nota_media_1sem"].isna().sum())],
        "na_2sem": [int(df_clean["nota_media_2sem"].isna().sum())],
    }
)

df_clean = tratar_ceros_notas_sin_evaluacion(df_clean)

despues = pd.DataFrame(
    {
        "zeros_1sem": [int((df_clean["nota_media_1sem"] == 0).sum())],
        "zeros_2sem": [int((df_clean["nota_media_2sem"] == 0).sum())],
        "na_1sem": [int(df_clean["nota_media_1sem"].isna().sum())],
        "na_2sem": [int(df_clean["nota_media_2sem"].isna().sum())],
    }
)

pd.concat({"antes": antes, "despues": despues})

Out[5]: 
           zeros_1sem  zeros_2sem  na_1sem  na_2sem
antes   0         718         870        0        0
despues 0         369         469      349      401


## Mezclar filas

Barajo las filas con una semilla fija para no depender del orden original del CSV.


In [6]:
df_base = permutar_filas(df_clean, random_state=42)
df_base.head()

Out[6]: 
   estado_civil               modo_solicitud  orden_solicitud  \
0  divorciado/a             mayor_de_23_anos                1   
1     soltero/a  2a_fase_contingente_general                1   
2     soltero/a  2a_fase_contingente_general                1   
3     soltero/a  2a_fase_contingente_general                2   
4     soltero/a             mayor_de_23_anos                1   

                               curso asistencia_diurna_vespertina  \
0                        equicultura                       diurna   
1                    servicio_social                       diurna   
2                   educacion_basica                       diurna   
3  gestion_de_publicidad_y_marketing                       diurna   
4                         enfermeria                       diurna   

   cualificacion_previa  nota_cualificacion_previa nacionalidad  \
0  educacion_secundaria                      133.1   portuguesa   
1  educacion_secundaria                      125.0 

## Tipos de columnas

Separo numericas y categoricas para construir despues los pipelines.


In [7]:
numeric_cols, categorical_cols = separa_tipos(df_base)

pd.DataFrame(
    {
        "tipo": ["numericas", "categoricas"],
        "n_columnas": [len(numeric_cols), len(categorical_cols)],
    }
)

Out[7]: 
          tipo  n_columnas
0    numericas          19
1  categoricas          18


In [8]:
print("Columnas numericas:")
print(numeric_cols)
print("\nColumnas categoricas:")
print(categorical_cols)

Columnas numericas:
['orden_solicitud', 'nota_cualificacion_previa', 'nota_admision', 'edad_al_matricularse', 'asignaturas_1sem_convalidadas', 'asignaturas_1sem_matriculadas', 'asignaturas_1sem_evaluadas', 'asignaturas_1sem_aprobadas', 'nota_media_1sem', 'asignaturas_1sem_sin_evaluacion', 'asignaturas_2sem_convalidadas', 'asignaturas_2sem_matriculadas', 'asignaturas_2sem_evaluadas', 'asignaturas_2sem_aprobadas', 'nota_media_2sem', 'asignaturas_2sem_sin_evaluacion', 'tasa_desempleo', 'tasa_inflacion', 'pib']

Columnas categoricas:
['estado_civil', 'modo_solicitud', 'curso', 'asistencia_diurna_vespertina', 'cualificacion_previa', 'nacionalidad', 'cualificacion_madre', 'cualificacion_padre', 'ocupacion_madre', 'ocupacion_padre', 'desplazado', 'necesidades_educativas_especiales', 'deudor', 'matricula_al_dia', 'genero', 'becado', 'internacional', 'objetivo']


## Datasets por tarea

Uso distintas vistas del dataset segun la tarea. No tiene sentido usar exactamente las mismas columnas para todo.


In [9]:
X_cls_early, y_cls_early = get_classification_dataset(df_base, stage="early")
X_cls_sem1, y_cls_sem1 = get_classification_dataset(df_base, stage="sem1")

pd.DataFrame(
    {
        "dataset": ["clasificacion_early", "clasificacion_sem1"],
        "n_filas": [len(X_cls_early), len(X_cls_sem1)],
        "n_features": [X_cls_early.shape[1], X_cls_sem1.shape[1]],
        "target": ["objetivo", "objetivo"],
    }
)

Out[9]: 
               dataset  n_filas  n_features    target
0  clasificacion_early     4424          24  objetivo
1   clasificacion_sem1     4424          30  objetivo


In [10]:
X_reg_early, y_reg_early = get_regression_dataset(df_base, stage="early")
X_reg_sem1, y_reg_sem1 = get_regression_dataset(df_base, stage="sem1")

pd.DataFrame(
    {
        "dataset": ["regresion_early", "regresion_sem1"],
        "n_filas": [len(X_reg_early), len(X_reg_sem1)],
        "n_features": [X_reg_early.shape[1], X_reg_sem1.shape[1]],
        "target": ["nota_media_2sem", "nota_media_2sem"],
    }
)

Out[10]: 
           dataset  n_filas  n_features           target
0  regresion_early     4023          24  nota_media_2sem
1   regresion_sem1     4023          30  nota_media_2sem


In [11]:
X_unsup_entry = get_unsupervised_dataset(df_base, profile="entry")
X_unsup_sem1 = get_unsupervised_dataset(df_base, profile="sem1")

pd.DataFrame(
    {
        "dataset": ["no_supervisado_entry", "no_supervisado_sem1"],
        "n_filas": [len(X_unsup_entry), len(X_unsup_sem1)],
        "n_features": [X_unsup_entry.shape[1], X_unsup_sem1.shape[1]],
    }
)

Out[11]: 
                dataset  n_filas  n_features
0  no_supervisado_entry     4424          24
1   no_supervisado_sem1     4424          30


### Comprobacion de fuga

En regresion no dejo entrar columnas del segundo semestre, porque el target tambien es del segundo semestre.


In [12]:
sem2_cols_presentes_en_regresion = [col for col in X_reg_sem1.columns if "2sem" in col]
sem1_cols_presentes_en_cls_early = [col for col in X_cls_early.columns if "1sem" in col]

pd.DataFrame(
    {
        "chequeo": [
            "columnas_2sem_en_regresion_sem1",
            "columnas_1sem_en_clasificacion_early",
        ],
        "resultado": [
            sem2_cols_presentes_en_regresion,
            sem1_cols_presentes_en_cls_early,
        ],
    }
)

Out[12]: 
                                chequeo resultado
0       columnas_2sem_en_regresion_sem1        []
1  columnas_1sem_en_clasificacion_early        []


## Orden del pipeline

El split se hace antes de imputar, escalar y aplicar one-hot. Asi el test no participa en el ajuste de esas transformaciones.


In [13]:
X = X_cls_sem1.copy()
y = y_cls_sem1.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

from src.utils import adaptar_dataframe_a_sklearn

X_train = adaptar_dataframe_a_sklearn(X_train)
X_test = adaptar_dataframe_a_sklearn(X_test)

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols),
    ]
)

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

pd.DataFrame(
    {
        "bloque": ["X_train", "X_test", "X_train_prepared", "X_test_prepared"],
        "shape": [
            X_train.shape,
            X_test.shape,
            X_train_prepared.shape,
            X_test_prepared.shape,
        ],
    }
)


Out[13]: 
             bloque        shape
0           X_train   (3539, 30)
1            X_test    (885, 30)
2  X_train_prepared  (3539, 238)
3   X_test_prepared   (885, 238)


## Resumen del preprocesado

- `en_blanco` pasa a missing.
- `desconocido` se mantiene como categoria.
- Los ceros en notas se tratan segun si hubo evaluaciones.
- No se eliminan filas por relaciones academicas raras.
- Cada tarea tiene sus columnas propias.
- Imputacion, escalado y one-hot se ajustan solo con train.
